# KAVACH — teach Llama 3.2 1B our scam playbook (LoRA) and make the phone file

**What this does:** downloads Llama 3.2 1B Instruct, trains a small add-on (LoRA) on KAVACH's
synthetic scam-call examples, checks the answers before and after, and converts the result to the
ExecuTorch `.pte` file the phone app loads. Everything is saved to your Google Drive under `kavach/`.

**Before you start**
1. Hugging Face account → open <https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct> → accept the licence (approval is usually minutes).
2. <https://huggingface.co/settings/tokens> → *Create new token* → type **Read** → copy it.
3. In Colab: **Runtime → Change runtime type → T4 GPU → Save**. Then **Runtime → Run all**.

You will be asked twice: allow Google Drive, and paste the token. Total time ≈ 1.5–2 h on a free T4.
Nothing here uses real people's calls; the data is generated by `data/synth/generate_calls.py`.
The locked held-out test split is **never** used for training or tuning.

In [ ]:
# 1) Check the GPU and connect Google Drive (results are copied there at each stage)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/kavach'
os.makedirs(OUT, exist_ok=True)
print('Results folder:', OUT)

In [ ]:
# 2) Get the KAVACH code and the training libraries
%cd /content
!rm -rf KAVACH && git clone -q --depth 1 https://github.com/siddhaantxsingh/KAVACH.git
%cd /content/KAVACH
!pip -q install -r models/llama/requirements-train.txt
!git log --oneline -1

In [ ]:
# 3) Hugging Face token (paste it; it is not printed or saved in the notebook)
from getpass import getpass
from huggingface_hub import login, hf_hub_download
tok = getpass('Paste your Hugging Face READ token: ').strip()
login(token=tok)
os.environ['HF_TOKEN'] = tok
BASE = 'meta-llama/Llama-3.2-1B-Instruct'
# Fails here with 403 if the licence was not accepted yet on the model page.
TOKENIZER = hf_hub_download(BASE, 'original/tokenizer.model')
print('Access OK. Tokenizer:', TOKENIZER)

In [ ]:
# 4) Make the training examples (synthetic calls -> question/answer pairs in the phone's exact prompt format)
!python data/synth/generate_calls.py
!python models/llama/make_sft_data.py
!wc -l data/synth/out/sft_train.jsonl data/synth/out/sft_dev.jsonl

In [ ]:
# 5) Score the ORIGINAL model first (so the report can show before vs after). ~8 min.
N_EVAL = 100
!python models/llama/eval_llama.py --hf {BASE} --n {N_EVAL} --name llama_base
!cp bench/results/llama_base.json {OUT}/

In [ ]:
# 6) Train the LoRA add-on. ~45-75 min on a T4. Saves the small adapter and a merged copy.
# Colab ships an old torchao that makes peft refuse to start; training does not need it.
!pip -q uninstall -y torchao
!python models/llama/finetune_lora.py --base {BASE} \
    --train data/synth/out/sft_train.jsonl --dev data/synth/out/sft_dev.jsonl \
    --out models/llama/work/kavach-lora --epochs 1
assert os.path.exists('models/llama/work/kavach-lora/merged/config.json'), 'Training failed: see the error above'
!cp -r models/llama/work/kavach-lora/adapter {OUT}/lora_adapter
print('Adapter saved to Drive (small). If anything later fails, this is the part worth keeping.')

In [ ]:
# 7) Score the TRAINED model on the same examples. ~8 min.
!python models/llama/eval_llama.py --hf models/llama/work/kavach-lora/merged --n {N_EVAL} --name llama_lora
assert os.path.exists('bench/results/llama_lora.json'), 'Scoring failed: see the error above'
!cp bench/results/llama_lora.json {OUT}/
import json
b = json.load(open('bench/results/llama_base.json')); l = json.load(open('bench/results/llama_lora.json'))
print(f"{'metric':16s} {'before':>8s} {'after':>8s}")
for k in ['json_valid','scam_acc','type_acc','stop_stage_acc','evidence_ok','copied_example','say_lang_ok','sec_per_example']:
    print(f"{k:16s} {b.get(k,'-'):>8} {l.get(k,'-'):>8}")

In [ ]:
# 8) Convert to the phone format. ExecuTorch MUST be 1.4.0 = the version inside the app.
#    (Installing it changes torch; that is fine because every step runs as a fresh `python` process.)
!pip -q install -r models/llama/requirements.txt
!python -c "import executorch, torch; print('executorch OK, torch', torch.__version__)"
W = 'models/llama/work'
!python models/llama/hf_to_meta.py {W}/kavach-lora/merged {W}/meta
!rm -rf {W}/kavach-lora/merged   # free disk/RAM before export
!python -m executorch.extension.llm.export.export_llm --config models/llama/configs/kavach_xnnpack_8da4w.yaml \
    +base.checkpoint={W}/meta/consolidated.00.pth +base.params={W}/meta/params.json \
    +export.output_name={W}/kavach_llama3_2_1b.pte
assert os.path.exists(f'{W}/kavach_llama3_2_1b.pte'), 'Export failed: see the error above'
!ls -la {W}/*.pte

In [ ]:
# 9) Copy the phone file + tokenizer to Drive and write a short summary
import shutil, datetime
pte = 'models/llama/work/kavach_llama3_2_1b.pte'
shutil.copy(pte, f'{OUT}/kavach_llama3_2_1b.pte')
shutil.copy(TOKENIZER, f'{OUT}/tokenizer.model')
json.dump({'base': BASE, 'epochs': 1, 'n_eval': N_EVAL, 'before': b, 'after': l,
           'pte_mb': round(os.path.getsize(pte)/1e6, 1), 'executorch': '1.4.0',
           'made': datetime.datetime.utcnow().isoformat() + 'Z'},
          open(f'{OUT}/lora_summary.json', 'w'), indent=2)
print(open(f'{OUT}/lora_summary.json').read())
print('''
DONE. In Google Drive > kavach you now have:
  kavach_llama3_2_1b.pte   (phone file, about 1 GB)
  tokenizer.model
  llama_base.json, llama_lora.json, lora_summary.json   (send these to Claude for the report)

Put the phone file on the phone (Git Bash, phone plugged in, from the KAVACH folder):
  scripts/push_models.sh --llama ~/Downloads/kavach_llama3_2_1b.pte ~/Downloads/tokenizer.model
To go back to the official model:  scripts/push_models.sh --official-spinquant
''')

### If something goes wrong
* **403 / "gated repo"** in step 3 → the licence is not approved yet on the model page; wait, then re-run from step 3.
* **"CUDA out of memory"** in step 6 → Runtime → Restart session, then run steps 1–4 and 6 again with `--max-len 1024` added.
* **Session crashed / ran out of RAM** in step 8 → the adapter is already safe in Drive (`kavach/lora_adapter`). Run steps 1–3, then the last cell below (rebuilds the merged model from the adapter), then step 8 again.

In [ ]:
# (Only if step 8 crashed) Rebuild the merged model from the adapter saved in Drive, then run step 8 again.
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
m = PeftModel.from_pretrained(m, f'{OUT}/lora_adapter').merge_and_unload()
m.save_pretrained('models/llama/work/kavach-lora/merged', safe_serialization=True)
print('merged model rebuilt')